# Drag Area Coefficient

**Author:** Ryan Huang

**Date:** 2026-05-16

**Relevant Links:**
- Add any DR0s, monday.com items, or other relevant links
- [Example Project DR0](https://example.com)

## Imports

Import any packages or dependencies for this project.
Uncomment or delete these lines as needed, and add other dependencies

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os
import dill
from data_tools.collections import TimeSeries
from data_tools.query import DBClient
from datetime import datetime
from pathlib import Path


## Data Acquisition

Obtain the data relevant to this analysis. Load data from saved files, or query it directly.

In [16]:
LOAD_FROM_FILE = False 
DATA_DIR = Path("data")

if LOAD_FROM_FILE:
    
    speed_file = DATA_DIR / "cruise_motor_speeds.bin"
    current_file = DATA_DIR / "cruise_pack_currents.bin"
    brake_file = DATA_DIR / "cruise_brake_presseds.bin"
    with open(speed_file, "rb") as f:
        cruise_motor_speeds: list[TimeSeries] = dill.load(f)
    with open(current_file, "rb") as f:
        cruise_pack_currents: list[TimeSeries] = dill.load(f)
    with open(brake_file, "rb") as f:
        cruise_brake_presseds: list[TimeSeries] = dill.load(f)

else:
    client = DBClient()
    PACK_CURRENT_THRESHOLD = 0.10682  # pack current never reads 0; its lowest value is this constant (constant is rounded up here)
    start_times = [datetime.fromisoformat("2026-04-02T20:02:30Z"),
                   datetime.fromisoformat("2026-04-02T20:07:10Z"),
                   datetime.fromisoformat("2026-04-02T20:26:05Z"),
                   datetime.fromisoformat("2026-04-02T20:35:40Z"),
                   datetime.fromisoformat("2026-04-02T21:05:25Z"),
                   datetime.fromisoformat("2026-04-02T21:10:30Z"),]
    
    stop_times = [datetime.fromisoformat("2026-04-02T20:03:10Z"),
                  datetime.fromisoformat("2026-04-02T20:07:30Z"),
                  datetime.fromisoformat("2026-04-02T20:26:24Z"),
                  datetime.fromisoformat("2026-04-02T20:36:15Z"),
                  datetime.fromisoformat("2026-04-02T21:05:40Z"),
                  datetime.fromisoformat("2026-04-02T21:10:40Z"),]
    
    cruise_motor_speeds: list[TimeSeries] = []
    cruise_pack_currents: list[TimeSeries] = []
    cruise_brake_presseds: list[TimeSeries] = []

    for start, stop in zip(start_times, stop_times):
        motor_rotating_speed: TimeSeries = client.query_time_series(start=start, stop=stop, field="MotorRotatingSpeed", units="km/h")
        pack_current: TimeSeries = client.query_time_series(start=start, stop=stop, field="PackCurrent", units="A")
        brake_pressed: TimeSeries = client.query_time_series(start=start, stop=stop, field="BrakePressed")

        cruise_start_index = np.where(pack_current <= PACK_CURRENT_THRESHOLD)[0][0]
        cruise_end_index = np.where(brake_pressed > 0)[0][0]

        cruise_start_time = pack_current.datetime_x_axis[cruise_start_index]
        cruise_end_time = brake_pressed.datetime_x_axis[cruise_end_index]
        
        cruise_motor_speeds.append(motor_rotating_speed.slice(cruise_start_time, cruise_end_time))
        cruise_pack_currents.append(pack_current.slice(cruise_start_time, cruise_end_time))
        cruise_brake_presseds.append(brake_pressed.slice(cruise_start_time, cruise_end_time))

    # with open(DATA_DIR / "cruise_motor_speeds.bin", 'wb') as f:
    #     dill.dump(cruise_motor_speeds, f)
    # with open(DATA_DIR / "cruise_pack_currents.bin", 'wb') as f:
    #     dill.dump(cruise_pack_currents, f)
    # with open(DATA_DIR / "cruise_brake_presseds.bin", 'wb') as f:
    #     dill.dump(cruise_brake_presseds, f)

## Analysis

Compute the desired results and present them. Take advantage of Markdown cells to guide the reader through your derivation process.

### Plotting the Data

In [ ]:
for i, (speed, current, brake) in enumerate(zip(cruise_motor_speeds, cruise_pack_currents, cruise_brake_presseds)):
    pass

### Analysis

In [ ]:
for i, speed in enumerate(cruise_motor_speeds):
    KMH_PER_MPS = 3.6
    acceleration = (np.diff(speed) / speed.period) / KMH_PER_MPS